# Applegate Mechanism Calculator
### A Hands-on Notebook Based on Völschow et al. (2016) & Navarrete et al. (2026)

*Prepared by Furkan Akar & Ozgur Basturk, November 2025 - June 2026*

In this notebook we implement and explore the energy requirements of the Applegate mechanism withiin the frame work given by  [Völschow et al. (2016)](https://ui.adsabs.harvard.edu/abs/2016A%26A...587A..34V/abstract) to derive the potential of magnetic mechanism to explain the Eclipse Timing Variations (ETV) observed in PCEB systems. We also employ [Azimutal Dynamo Weves model](https://ui.adsabs.harvard.edu/abs/2026arXiv260427609N/abstract) to determine whether the observed amplitudes in the ETV diagrams of PCEB systems can be regenerated by the azimutal component of the quadrupole moment changes.

We follow the formalism presented in

We will:

1. Define the relevant stellar and orbital parameters.
2. Implement three different models for the Applegate mechanism:
   - Thin-shell approximation (Tian et al. 2009, as quoted by Völschow et al. 2016)
   - Constant-density model
   - Two-zone model
3. Compare the required energy to the energy budget of the secondary star.
4. Determine the potential amplitudes of the ETVs due to ADW mechanism.
5. Compare these amplitudes to the observed amplitudes in the ETV systems.

## 1. Physical Constants and Units

All equations in Völschow et al. (2016) are expressed in cgs units. We make use of
[Astropy-Constants](https://docs.astropy.org/en/stable/constants/index.html) to obtain physical constants and convert them to cgs.

We also adopt the IAU 2015-Resolution B3 nominal solar effective temperature:
$T_\odot = 5772 ~\mathrm{K}.$

In [1]:
from astropy.constants import G, M_sun, R_sun, L_sun
from astropy import units as u
import numpy as np
import math

G_cgs = G.cgs.value
M_sun_cgs = M_sun.cgs.value
R_sun_cgs = R_sun.cgs.value
L_sun_cgs = L_sun.cgs.value

day_to_sec = u.day.to(u.s)
year_to_sec = u.year.to(u.s)

T_sun = 5772  # Kelvin #  IAU 2015 Resolution B3

## 2. Input Parameters for the Secondary Star and the Binary

The parameters of a Post Common Envelope Binary System that are considered are for

- A white dwarf primary, and
- A low-mass main-sequence secondary star with:
  - mass $M_{\mathrm{sec}}$ (in units of $M_\odot$),
  - radius $R_{\mathrm{sec}}$ (in units of $R_\odot$),
  - effective temperature $T_{\mathrm{sec}}$ (in Kelvin),
  - luminosity $L_{\mathrm{sec}}$ (in units of $L_\odot$).

The binary is characterized by:

- binary separation $a_{\mathrm{bin}}$ (in units of $R_\odot$),
- orbital period $P_{\mathrm{bin}}$ (in days),
- modulation period $P_{\mathrm{mod}}$ of the ETV signal (in days or years),
- observed relative period change $\Delta P / P_{\mathrm{bin}}$ (dimensionless),
  or equivalently a timing semi-amplitude $A$ in seconds.

## 3. Thin-Shell Approximation (Tian et al. 2009)

In the thin-shell approximation, Tian et al. (2009) (quoted as Eq. (7) in Völschow et al. 2016)
derived an approximate relation between the required energy and the observed timing variations:

$$
\frac{\Delta E}{E_{\mathrm{sec}}}
= 0.233
\left( \frac{M_{\mathrm{sec}}}{M_{\odot}} \right)^{3}
\left( \frac{R_{\mathrm{sec}}}{R_{\odot}} \right)^{-10}
\left( \frac{T_{\mathrm{sec}}}{6000~\mathrm{K}} \right)^{-4}
\left( \frac{a_{\mathrm{bin}}}{R_{\odot}} \right)^{4}
\left( \frac{\Delta P}{\mathrm{s}} \right)^{2}
\left( \frac{P_{\mathrm{mod}}}{\mathrm{yr}} \right)^{-1}
$$

where $\Delta P$ is the absolute period change in seconds, and $P_{\mathrm{mod}}$ is the modulation period in years. $E_{\mathrm{sec}}$ is the energy produced by the secondary over one modulation period.

In [2]:
def thin_shell_energy_ratio(M_sec, R_sec, T_sec, a_bin, delta_P_sec, P_mod_yr):
    ratio = 0.233 * (M_sec**3) * (R_sec**-10) * (T_sec / 6000)**(-4) * (a_bin**4) * (delta_P_sec**2) * (P_mod_yr**-1)
    return ratio

## 4. Constant-Density Model (Völschow et al. 2016, Eq. 22)

Assuming a constant density profile for the secondary star, Völschow et al. (2016) derive for the required energy (their Eq. (22)):

$$
\frac{\Delta E}{E_{\mathrm{sec}}}
\simeq 1.1 \times 10^{7}
\left( \frac{\Delta P}{P_{\mathrm{bin}}} \right)
\left( \frac{a_{\mathrm{bin}}}{R_{\odot}} \right)^{2}
\left( \frac{M_{\mathrm{sec}}}{M_{\odot}} \right)^{2}
\left( \frac{R_{\mathrm{sec}}}{R_{\odot}} \right)^{-3}
\left( \frac{P_{\mathrm{mod}}}{\mathrm{yr}} \right)^{-1}
\left( \frac{L_{\mathrm{sec}}}{L_{\odot}} \right)^{-1}
$$

This expression already includes the division by the energy available over one modulation period, so it directly yields $\Delta E / E_{\mathrm{sec}}$.

In [3]:
def constant_density_energy_ratio(deltaP_over_Pbin, a_bin, M_sec, R_sec, P_mod_yr, L_sec):
    ratio = 1.1e7 * deltaP_over_Pbin * (a_bin**2) * (M_sec**2) * (R_sec**-3) * (P_mod_yr**-1) * (L_sec**-1)
    return ratio

## 5. Two-Zone Model (Völschow et al. 2016, Eq. 35–37)

The two-zone model represents the secondary star as an inner core and an outer shell with different
densities. The exchange of angular momentum between these two regions changes the stellar quadrupole
moment and therefore the orbital period. The required energy is given by:

$$
\frac{\Delta E}{E_{\mathrm{sec}}}
= k_{1}\,
\frac{M_{\mathrm{sec}} R_{\mathrm{sec}}^{2}}
     {P_{\mathrm{bin}}^{2}\, P_{\mathrm{mod}}\, L_{\mathrm{sec}}}
\left(
    1 \pm
    \sqrt{
        1 -
        k_{2}\, G\,
        \frac{
            a_{\mathrm{bin}}^{2}\,
            M_{\mathrm{sec}}\,
            P_{\mathrm{bin}}^{2}
        }{
            R_{\mathrm{sec}}^{5}
        }
        \frac{\Delta P}{P_{\mathrm{bin}}}
    }
\right)^{2}
$$


The constants $k_1$ and $k_2$ are structural coefficients of the two-zone approximation.

- $k_1$ is a structural coefficient that measures how efficiently a given exchange of angular momentum between core and shell can change the stellar quadrupole moment. It depends on the relative moments of inertia of the core and the shell.

- $k_2$ determines how strongly a change in the quadrupole moment couples back into the orbital period. It effectively controls the strength of the interaction between the stellar deformation and the binary orbit.

For low-mass main-sequence secondaries typical of post-common-envelope binaries, Völschow et al. (2016) obtain:

$$
k_1 = 0.133, \qquad k_2 = 3.42
$$

The term under the square root is what Völschow et al. (2016) refer to as the Applegate parameter:

$$
A = k_{2}\, G\,
\frac{
    a_{\mathrm{bin}}^{2}\,
    M_{\mathrm{sec}}\,
    P_{\mathrm{bin}}^{2}
}{
    R_{\mathrm{sec}}^{5}
}
\left(
    \frac{\Delta P}{P_{\mathrm{bin}}}
\right)
$$

A physical solution exists only if $(A \le 1)$. If $(A > 1)$, the Applegate mechanism cannot produce the observed period variations.

In [4]:
k1 = 0.133
k2 = 3.42

def two_zone_energy_ratio(M_sec, R_sec, L_sec, a_bin, P_bin_days, deltaP_over_Pbin, P_mod_days, G_value=G_cgs):
    
    a_cgs = a_bin * R_sun_cgs
    M_cgs = M_sec * M_sun_cgs
    R_cgs = R_sec * R_sun_cgs
    L_cgs = L_sec * L_sun_cgs
    
    P_bin_sec = P_bin_days * day_to_sec
    P_mod_sec = P_mod_days * day_to_sec
    
    # Applegate parameter A (their Eq. 37)
    A = k2 * G_value * (a_cgs**2 * P_bin_sec**2 * M_cgs / R_cgs**5) * deltaP_over_Pbin

    if A > 1.0:
        return None, A

    prefactor = k1 * (M_cgs * R_cgs**2) / (P_bin_sec**2 * P_mod_sec * L_cgs)
    ratio = prefactor * (1.0 - math.sqrt(1.0 - A))**2

    return ratio, A

## 6. Application to Example Systems from Völschow et al. (2016)

We now apply the three analytical models

1. Thin-shell approximation (Tian et al. 2009, Eq. 7),
2. Constant-density model (Völschow et al. 2016, Eq. 22),
3. Two-zone model (Völschow et al. 2016, Eq. 35–37),

to three well-studied systems from Völschow et al. (2016): **NN Ser**, **HW Vir**, and **QS Vir**.

For each system we adopt the stellar and orbital parameters listed in their Tables 1 and 2, and the
period modulation parameters from Table 3 (i.e. modulation period and $\Delta P / P_{\mathrm{bin}}$). We then compute $\Delta E / E_{\mathrm{sec}}$ for all three models and compare them to the values reported in Table 4 of the paper.

In [5]:
# Parameters for three systems from Völschow et al. (2016)
# NN Ser, HW Vir, QS Vir
systems = {
    "NN Ser": {"M_sec": 0.111,
               "R_sec": 0.149,
               "T_sec": 2920,
               "L_sec": 0.00147,
               "a_bin": 0.934,
               "P_bin_days": 0.130,     # days
               "P_mod_yr": 15.482,      # years
               "deltaP_over_Pbin": 7.1e-7},
    
    "HW Vir": {"M_sec": 0.142,
               "R_sec": 0.175,
               "T_sec": 3084,
               "L_sec": 0.003,
               "a_bin": 0.860,
               "P_bin_days": 0.117,     # days
               "P_mod_yr": 55,          # years
               "deltaP_over_Pbin": 4.1e-6},
    
    "QS Vir": {"M_sec": 0.43,
               "R_sec": 0.42,
               "T_sec": 3100.0,
               "L_sec": 0.0146,
               "a_bin": 1.27,
               "P_bin_days": 0.151,     # days
               "P_mod_yr": 16.99,       # years
               "deltaP_over_Pbin": 1.0e-6}
}

In [6]:
for name, p in systems.items():
    M = p["M_sec"]
    R = p["R_sec"]
    T = p["T_sec"]
    L = p["L_sec"]
    a = p["a_bin"]
    
    P_bin_days = p["P_bin_days"]
    P_bin_sec = P_bin_days * day_to_sec
    
    P_mod_yr = p["P_mod_yr"]
    P_mod_days = P_mod_yr * 365.25
    
    deltaP_over_Pbin = p["deltaP_over_Pbin"]
    delta_P_sec = deltaP_over_Pbin * P_bin_sec

    ratio_thin = thin_shell_energy_ratio(M_sec=M, 
                                         R_sec=R, 
                                         T_sec=T, 
                                         a_bin=a, 
                                         delta_P_sec=delta_P_sec, 
                                         P_mod_yr=P_mod_yr)

    ratio_const = constant_density_energy_ratio(deltaP_over_Pbin=deltaP_over_Pbin, 
                                                a_bin=a, 
                                                M_sec=M, 
                                                R_sec=R, 
                                                P_mod_yr=P_mod_yr, 
                                                L_sec=L)

    ratio_two, A = two_zone_energy_ratio(M_sec=M, 
                                         R_sec=R, 
                                         L_sec=L, 
                                         a_bin=a, 
                                         P_bin_days=P_bin_days, 
                                         deltaP_over_Pbin=deltaP_over_Pbin, 
                                         P_mod_days=P_mod_days)

    print(f"\u2022 {name}")
    print(f"  {'A parameter':20s}: {A:.3f}")
    print(f"  {'Thin-shell':20s}: dE/E_sec ~= {ratio_thin:.2f}")
    print(f"  {'Constant-density':20s}: dE/E_sec ~= {ratio_const:.2f}")
    print(f"  {'Two-zone':20s}: dE/E_sec ~= {ratio_two:.2f}")

• NN Ser
  A parameter         : 0.159
  Thin-shell          : dE/E_sec ~= 3.29
  Constant-density    : dE/E_sec ~= 1115.03
  Two-zone            : dE/E_sec ~= 62.72
• HW Vir
  A parameter         : 0.361
  Thin-shell          : dE/E_sec ~= 6.06
  Constant-density    : dE/E_sec ~= 760.59
  Two-zone            : dE/E_sec ~= 110.25
• QS Vir
  A parameter         : 0.012
  Thin-shell          : dE/E_sec ~= 0.04
  Constant-density    : dE/E_sec ~= 178.50
  Two-zone            : dE/E_sec ~= 0.71


## 7. Interpretation of the Results

Using the analytical expressions reproduced in this notebook, we compute $\Delta E / E_{\mathrm{sec}}$ for the systems NN Ser, HW Vir, and QS Vir. Our results reproduce the values reported in Table 4 of Völschow et al. (2016) to within a few percent, confirming that our implementation is correct.

### Consistency with Völschow et al. (2016)

- **NN Ser**  
  - Thin-shell: 3.29  
  - Constant-density: 1115.03  
  - Two-zone: 62.72  
  These values are in excellent agreement with the published values  
  (3.3, 1100, 64).  

- **HW Vir**  
  - Thin-shell: 6.06 
  - Constant-density: 760.59  
  - Two-zone: 110.25
  These values are again consistent with the reported values  
  (6.0, 720, 108).  

- **QS Vir**  
  - Thin-shell: 0.04  
  - Constant-density: 178.50  
  - Two-zone: 0.71  
  Matching the values  
  (0.040, 170, 0.71).

These comparisons verify that our thin-shell, constant-density, and two-zone implementations reproduce the analytical models described by Völschow et al. (2016)

---

## 8. Astrophysical Interpretation

$\Delta E / E_{\mathrm{sec}}$ measures the energy required to produce the observed period modulation relative to the total energy generated by the secondary during one modulation cycle.

- **ΔE/E_sec ≪ 1** → energetically very easy; Applegate mechanism is viable  
- **ΔE/E_sec ∼ 1** → borderline case; energetically possible  
- **ΔE/E_sec ≫ 1** → energetically impossible; would require more energy than the star can supply  

### NN Ser  
Even in the most realistic approximation (two-zone), $\Delta E / E_{\mathrm{sec}} \approx 60$ meaning the required energy exceeds the available energy by almost two orders of magnitude. The Applegate mechanism is energetically unfeasible for NN Ser, strongly supporting a third-body interpretation for its timing variations.

### HW Vir  
Similarly, HW Vir yields $\Delta E/E_{\mathrm{sec}} \approx 110$ again far above unity. The Applegate mechanism is rejected on energetic grounds for HW Vir as well.

### QS Vir  
QS Vir is fundamentally different: its two-zone value $\Delta E/E_{\mathrm{sec}} \approx 0.7$ is of order unity. This means the magnetic quadrupole mechanism is energetically viable for QS Vir and cannot be ruled out. This system remains a borderline case in which both Applegate-type magnetic cycles and third-body scenarios should be considered.

---

In summary, our results confirm the findings of Völschow et al. (2016): most systems require too much energy for an Applegate-type mechanism, except for a few borderline cases such as QS Vir.

## 9. Application to NY Vir using Esmer et al. (2023)

As a final step, we follow the procedure detailed above and implement it to the **NY Vir** system using the parameters reported by [Esmer et al. (2023)](https://ui.adsabs.harvard.edu/abs/2023MNRAS.525.6050E/abstract). Their study also evaluates the required energy for the **Thin–shell**, **Constant–density**, and **Two–zone** models based on Völschow implementation run through a web interface, which is not online anymore!

We repeat the calculations below and compare the results.

In [7]:
# Parameters for NY Vir from Esmer et al. (2023)

M2   = 0.13       # M_sun
R2   = 0.155      # R_sun
T2   = 2048       # K

a_bin = 0.77           # R_sun
P_bin = 0.1010159690   # days

P_mod_days = 8127.15    # days
A_OC_sec   = 35.1       # s

In [8]:
L2 = (R2**2) * (T2 / T_sun)**4

P_mod_sec = P_mod_days * day_to_sec
P_mod_yr  = P_mod_days / 365.25

deltaP_over_P = 2.0 * math.pi * A_OC_sec / P_mod_sec
deltaP_sec    = deltaP_over_P * P_bin * day_to_sec


ratio_thin = thin_shell_energy_ratio(M2, R2, T2, a_bin, deltaP_sec, P_mod_yr)

ratio_const = constant_density_energy_ratio(deltaP_over_P, a_bin, M2, R2, P_mod_yr, L2)

ratio_two, A = two_zone_energy_ratio(M2, R2, L2, a_bin, P_bin, deltaP_over_P, P_mod_days)

print(f"\u2022 NY Vir (Esmer et al. 2023) ")
print(f"  {'A parameter':20s}: {A:.3f}")
print(f"  {'Thin-shell':20s}: dE/E_sec ~= {ratio_thin:.3f}")
print(f"  {'Constant-density':20s}: dE/E_sec ~= {ratio_const:.3f}")
print(f"  {'Two-zone':20s}: dE/E_sec ~= {ratio_two:.3f}")

• NY Vir (Esmer et al. 2023) 
  A parameter         : 0.028
  Thin-shell          : dE/E_sec ~= 0.559
  Constant-density    : dE/E_sec ~= 1097.172
  Two-zone            : dE/E_sec ~= 10.035


Using the parameters reported by Esmer et al. (2023) for NY Vir, our implementation reproduces the published values with good agreement:

- Thin–shell model: $\Delta E / E_{\mathrm{sec}} = 0.559$  
- Constant–density model: $\Delta E / E_{\mathrm{sec}} = 1097.17$  
- Two–zone model: $\Delta E / E_{\mathrm{sec}} = 10.035$

These values match the results presented in Esmer et al. (2023):  
$\Delta E / E_{\mathrm{sec}} = 0.559$, $1101.75$, and $10.07$ for the thin–shell, constant–density, and two–zone models, respectively. The small numerical differences between our results and those reported by Esmer et al. (2023) are expected and most likely arise from the specific choices of physical constants used in the calculations, in particular the adopted solar temperature and related luminosity scaling.

From a physical standpoint, the interpretation follows directly:

- The constant–density model predicts extremely large energy requirements ($\sim 10^3$), well beyond the secondary star's capability.  
- The two–zone model also yields a value $(\sim 10$), meaning the star would need to devote more than ten times its available energy to drive the modulation - not physically plausible.  
- Only the thin–shell model gives a value below unity, but still large ($\sim 0.56$), implying that more than half of the secondary’s luminosity would need to be channelled into quadrupole moment oscillations.

Therefore, even in the most optimistic scenario (thin–shell approximation), the energy requirement remains uncomfortably high.  
This supports the conclusion of Esmer et al. (2023) that the Applegate mechanism is unlikely to be the primary cause of the observed ETV modulation in NY Vir.

## Conclusion

In this notebook, we implement three formulations of the Applegate mechanism—the thin–shell, constant–density, and two–zone models—following the formalisms presented by Völschow et al. (2016) and Esmer et al. (2023). By applying these models to the PCEB systems, NN Ser, HW Vir, QS Vir, and NY Vir, we successfully reproduce the energy budget values reported in the literature. This agreement validates the accuracy of the implementation and confirms the reliability of the code for investigation of the Applegate mechanism across different systems.

From an astrophysical standpoint, for NN Ser and HW Vir all models require $\Delta E/E_{\mathrm{sec}} \gg 1$, confirming that the mechanism is energetically impossible. QS Vir remains the only borderline case: its two–zone value of $\sim 0.7$ indicates that magnetic quadrupole variations cannot be ruled out.

Applying the same analysis to NY Vir using the parameters from Esmer et al. (2023), the results again show that the required energy—especially $\Delta E/E_{\mathrm{sec}} \approx 10$ from the two–zone model—is far above what the secondary can supply.

In summary, while QS Vir may marginally allow Applegate-type activity, the mechanism is not energetically viable for NN Ser, HW Vir, or NY Vir, reinforcing the conclusion that magnetic cycles cannot explain their long-term eclipse timing variations.

## 10. Application to DD CrB using Baştürk et al. (2026)

Another example is given below for DD CrB, which <a href="https://ui.adsabs.harvard.edu/abs/2026MNRAS.547ag290B/abstract">was published in 2026</a>. 

In [9]:
# Parameters for DD CrB from Baştürk et al. (2026)

M2   = 0.127       # M_sun
R2   = 0.1619      # R_sun
T2   = 2357        # K

a_bin = 1.020           # R_sun
P_bin = 0.161770446     # days

P_mod_days = 5100.81    # days
A_OC_sec   = 9.024306   # s

In [10]:
L2 = (R2**2) * (T2 / T_sun)**4

P_mod_sec = P_mod_days * day_to_sec
P_mod_yr  = P_mod_days / 365.25

deltaP_over_P = 2.0 * math.pi * A_OC_sec / P_mod_sec
deltaP_sec    = deltaP_over_P * P_bin * day_to_sec


ratio_thin = thin_shell_energy_ratio(M2, R2, T2, a_bin, deltaP_sec, P_mod_yr)

ratio_const = constant_density_energy_ratio(deltaP_over_P, a_bin, M2, R2, P_mod_yr, L2)

ratio_two, A = two_zone_energy_ratio(M2, R2, L2, a_bin, P_bin, deltaP_over_P, P_mod_days)

print(f"\u2022 DD CrB (Baştürk et al. 2026) ")
print(f"  {'A parameter':20s}: {A:.3f}")
print(f"  {'Thin-shell':20s}: dE/E_sec ~= {ratio_thin:.3f}")
print(f"  {'Constant-density':20s}: dE/E_sec ~= {ratio_const:.3f}")
print(f"  {'Two-zone':20s}: dE/E_sec ~= {ratio_two:.3f}")

• DD CrB (Baştürk et al. 2026) 
  A parameter         : 0.040
  Thin-shell          : dE/E_sec ~= 0.406
  Constant-density    : dE/E_sec ~= 549.826
  Two-zone            : dE/E_sec ~= 7.331


## 11. Azimuthal Dynamo Wave Model

To evaluate the feasibility of the Azimuthal Dynamo Wave (ADW) mechanism as the primary driver of the observed eclipse timing variations (ETVs), we followed the procedure as defined by <a href="https://arxiv.org/abs/2604.27609">Navarrete et al. (2026)</a>. We estimated the expected theoretical $O-C$ amplitudes using an analytical scaling framework. Assuming the system has reached tidal synchronization, the secondary star's rotation period is fixed to the binary orbital period ($P_{orb} \simeq 0.136$ days), yielding a linear rotational velocity and an angular rotation rate based on the value. In the highly rapid rotation regime of PCEBs with very short orbital periods, an $\alpha^2$ dynamo is expected to dominate, natively producing strong, azimuthally migrating non-axisymmetric magnetic fields. To approximate the resulting non-axisymmetric density perturbations, we adopted the baseline inertia tensor variations ($\Delta I_{xx}$, $\Delta I_{yy}$, $\Delta I_{zz}$) from the most rapidly rotating fully convective 3D MHD model available in the current literature (Model A at $10\Omega_\odot$) from Table-3 in <a href="https://arxiv.org/abs/2604.27609">Navarrete et al. (2026)</a>.  

These variations in the principal moments of inertia were mapped to the variation in the non-axisymmetric quadrupole moment in the orbital plane using the linear substitution in Equation-20 of <a href="https://arxiv.org/abs/2604.27609">Navarrete et al. (2026)</a>


$$\Delta Q_{xx} = \Delta I_{xx} - \frac{1}{3}(\Delta I_{xx} + \Delta I_{yy} + \Delta I_{zz})$$

The theoretically derived $\Delta Q_{xx}$ was then scaled by the binary separation ($a_{\rm bin}$), the active star's mass, and the empirically observed modulation periods to predict the semi-amplitude of the timing variations via the analytical approximation si given with their Equation-24 as  

$$O-C = \frac{9}{2\pi} \frac{\Delta Q_{xx}}{a^2 M_2} P_{mod}$$ 

Applying this framework yields expected theoretical $O-C$ amplitudes for the O-C variations. These theoretical limits are then compared to those extracted from observed ETV, either through a frequency analysis such as that using Lomb-Scargle periodogram or sinusoidal fits to the observed ETV diaygrams. These comparisons demonstrate whether ADW mechanism provides a physically viable and energetically consistent explanation for the system's observed timing variations.

In [11]:
from math import *
from astropy import constants as const
def rotation_rate(R2,P_bin):
    # linear rotation rate
    Prot_sun = 25.05 # days (Snodgrass 1984, Snodgrass & Howard, 1985)
    Prot2 = P_bin / 86400 # tidal locking assumption
    omega_lin = (2*pi*R2*const.R_sun) / (P_bin)
    omega_ratio = (2*pi / Prot2) / (2*pi / Prot_sun)
    return(omega_lin,omega_ratio)
def quadrupole_moment(a_bin,M2,P_mod):
    # Table-3 reference values for the inertia tensor for 10 omega_sol
    delta_I_xx = 6.00192126e40
    delta_I_yy = 6.00267489e40
    delta_I_zz = 1.40925322e41
    delta_Q_xx = delta_I_xx - (delta_I_xx + delta_I_yy + delta_I_zz) / 3
    amp_OC = 9/(2*pi)*(delta_Q_xx/((a_bin*const.R_sun)**2*M2*const.M_sun)*P_mod)
    return(delta_Q_xx,amp_OC)

In [12]:
for name, p in systems.items():
    M = p["M_sec"]
    a = p["a_bin"]
    P = p["P_bin_days"]
    R = p["R_sec"]
    P_mod_yr = p["P_mod_yr"]
    P_mod_sec = P_mod_yr*365.25*86400
    P_bin_sec = P*86400
    omega_lin,omega_ratio = rotation_rate(R,P_bin_sec)
    delta_Q_xx, amp_OC = quadrupole_moment(a,M,P_mod_sec)   
    print(f"\u2022 {name}")
    print(f"The rotation rate of the active star is {omega_lin.value/1e3:.2f} km/s")
    print(f"The angular rotation rate of the active star in units of solar roation is {omega_ratio:.2f}")
    print(f"Change in the quadruple moment in the x-direction is Delta_Q_xx = {delta_Q_xx:.2e}") 
    print(f"O-C expected from the observed period in the O-C = {amp_OC:.2f} seconds")

• NN Ser
The rotation rate of the active star is 57.99 km/s
The angular rotation rate of the active star in units of solar roation is 192.69
Change in the quadruple moment in the x-direction is Delta_Q_xx = -2.70e+40
O-C expected from the observed period in the O-C = -202.55 1 / (kg m2) seconds
• HW Vir
The rotation rate of the active star is 75.67 km/s
The angular rotation rate of the active star in units of solar roation is 214.10
Change in the quadruple moment in the x-direction is Delta_Q_xx = -2.70e+40
O-C expected from the observed period in the O-C = -663.43 1 / (kg m2) seconds
• QS Vir
The rotation rate of the active star is 140.72 km/s
The angular rotation rate of the active star in units of solar roation is 165.89
Change in the quadruple moment in the x-direction is Delta_Q_xx = -2.70e+40
O-C expected from the observed period in the O-C = -31.03 1 / (kg m2) seconds


The negative sign in the change is not a math error; it is a real physical result of the dynamo simulations! The authors explicitly point this out in Appendix A. As the rotation rate increases from Model A ($10\Omega_\odot$) to Models B and C ($20\Omega_\odot$ and $30\Omega_\odot$), there is a "flip in the sign of the three components of Q".  The authors explain that this sign flip happens "due to increasingly strong magnetic fields that concentrate near the poles" in the more rapidly rotating models. This polar concentration alters the non-axisymmetric density distribution ($\rho'$), essentially making the dominant mass deformations prolate (cigar-shaped) rather than oblate (pancake-shaped) relative to the rotational axis.

Since we did not include $f$ (the scaling parameter in Table-2), we effectively used $f=1.0$, meaning the simulation results are based on the absolute weakest dynamo configuration. Even with this extreme underestimation, we still achieve $O-C$ amplitudes that match the observations for these systems. This actually makes the case even stronger: if the weakest possible ADW model can account for the observed ETVs, a realistically scaled model will have absolutely no problem doing so.

One fix for this can be to keep the current Table 3 methodology but simply multiply the $\Delta I$ values by $f=2.0$ (the maximum scaling factor used by the authors for Set sin2p0). Because Equation 24 is linear, this will simply double the expected $O-C$ amplitudes, firmly covering the upper bounds of your observed variations. 

<u>Adopting different scaling ($f$) and reference inertial ($\Delta I$) to the code is a work in progress!</u>

In the top-right panel (Model C graph in Appendix-A, Fig.A.2. for $30\Omega_\odot$), the $Q_{xx}$ value fluctuates between approximately $-2.475 \times 10^{43}$ and $-2.455 \times 10^{43} \text{ kg m}^2$. This gives a peak-to-peak variation amplitude ($\Delta Q_{xx}$) of roughly $0.01 \times 10^{43}$ (or $10 \times 10^{40} \text{ kg m}^2$). This true Model C variation is nearly four times larger than the unscaled Model A variation we calculated!  

# Citation:

Please cite <a href="https://ui.adsabs.harvard.edu/abs/2026MNRAS.547ag290B/abstract">DD CrB work from our group</a> if you use this code for the computation of energy requirements for Applegate mechanism. 

You can use the computations for the ADW mechanism at your own risk. Our paper on DW UMa is in preparation for the moment, some of the results of which are based on the computation with this mechanism. But that part of of the code is a work in progress!